# QuantJourney SDK - CCXT Cryptocurrency Data

This notebook demonstrates CCXT connector:
- Real-time crypto prices from 100+ exchanges
- OHLCV candlestick data
- Multi-exchange comparison
- Order book and trades

**API:** https://api.quantjourney.cloud

In [6]:
# Setup
import sys
sys.path.insert(0, '..')

from quantjourney.sdk import QuantJourneyAPI
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "png"

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# API Key authentication
import os
API_KEY = os.environ.get("QJ_API_KEY", "qj_...")
qj = QuantJourneyAPI(api_key=API_KEY)
print("✓ Connected to QuantJourney API")


✓ Connected to QuantJourney API


## 1. Bitcoin OHLCV Data

In [ ]:
# Fetch Bitcoin OHLCV from Binance
try:
    response = qj.ccxt.get_ohlcv_data(
        exchange="binance",
        symbol="BTC/USDT",
        timeframe="1d",
        limit=365
    )
    btc = response.get('value', response) if isinstance(response, dict) else response
    
    if btc:
        df_btc = pd.DataFrame(btc, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
        df_btc['date'] = pd.to_datetime(df_btc['timestamp'], unit='ms')
        df_btc = df_btc.sort_values('date')
        
        print(f"BTC records: {len(df_btc)}")
        print(f"Date range: {df_btc['date'].min().date()} to {df_btc['date'].max().date()}")
        print(f"\nLatest data:")
        print(df_btc[['date', 'open', 'high', 'low', 'close', 'volume']].tail())
except Exception as e:
    print(f"Note: CCXT connector - {e}")


In [2]:
# Plot Bitcoin candlestick chart
if 'df_btc' in dir() and len(df_btc) > 0:
    fig = go.Figure(data=[
        go.Candlestick(
            x=df_btc['date'],
            open=df_btc['open'],
            high=df_btc['high'],
            low=df_btc['low'],
            close=df_btc['close'],
            name='BTC'
        )
    ])
    
    fig.update_layout(
        title='Bitcoin (BTC/USDT) - Daily Candlestick',
        yaxis_title='Price (USDT)',
        xaxis_rangeslider_visible=False,
        template='plotly_dark',
        height=500
    )
    fig.show()
    
    print(f"\nBTC Statistics:")
    print(f"  Current:  ${df_btc['close'].iloc[-1]:,.0f}")
    print(f"  52w High: ${df_btc['high'].max():,.0f}")
    print(f"  52w Low:  ${df_btc['low'].min():,.0f}")
    print(f"  Avg Vol:  {df_btc['volume'].mean():,.0f} BTC")


## 2. Multi-Crypto Comparison

In [3]:
# Compare multiple cryptocurrencies
cryptos = ['BTC/USDT', 'ETH/USDT', 'SOL/USDT', 'BNB/USDT', 'XRP/USDT']
crypto_data = {}

print("Fetching crypto data...")
for symbol in cryptos:
    try:
        response = qj.ccxt.get_ohlcv_data(
            exchange="binance",
            symbol=symbol,
            timeframe="1d",
            limit=180
        )
        data = response.get('value', response) if isinstance(response, dict) else response
        if data:
            df = pd.DataFrame(data, columns=['ts', 'o', 'h', 'l', 'c', 'v'])
            df['date'] = pd.to_datetime(df['ts'], unit='ms')
            df = df.set_index('date')
            crypto_data[symbol.split('/')[0]] = df['c']
            print(f"  ✓ {symbol}: ${df['c'].iloc[-1]:,.2f}")
    except Exception as e:
        print(f"  ✗ {symbol}: {e}")


Fetching crypto data...


In [4]:
# Normalized performance comparison
if crypto_data:
    crypto_df = pd.DataFrame(crypto_data).dropna()
    
    # Normalize to 100
    normalized = crypto_df / crypto_df.iloc[0] * 100
    
    fig = px.line(
        normalized,
        title='Crypto Performance Comparison (Normalized to 100)',
        labels={'value': 'Performance', 'variable': 'Crypto'},
        template='plotly_dark'
    )
    fig.update_layout(height=500)
    fig.show()
    
    # Performance summary
    print("\n180-Day Performance:")
    for coin in normalized.columns:
        perf = normalized[coin].iloc[-1] - 100
        print(f"  {coin:>5}: {perf:+.1f}%")


## 3. Current Prices (Ticker)

In [5]:
# Get current ticker data
symbols = ['BTC/USDT', 'ETH/USDT', 'SOL/USDT']

print("Current Crypto Prices")
print("=" * 50)

for symbol in symbols:
    try:
        response = qj.ccxt.get_ticker(
            exchange="binance",
            symbol=symbol
        )
        ticker = response.get('value', response) if isinstance(response, dict) else response
        
        if ticker and isinstance(ticker, dict):
            last = ticker.get('last', ticker.get('close', 0))
            bid = ticker.get('bid', 0)
            ask = ticker.get('ask', 0)
            volume = ticker.get('baseVolume', ticker.get('volume', 0))
            change = ticker.get('percentage', ticker.get('change', 0))
            
            print(f"\n{symbol}:")
            print(f"  Last:   ${last:,.2f}")
            print(f"  Bid:    ${bid:,.2f}")
            print(f"  Ask:    ${ask:,.2f}")
            print(f"  24h Vol: {volume:,.0f}")
            if change:
                print(f"  24h Chg: {change:+.2f}%")
    except Exception as e:
        print(f"\n{symbol}: {e}")


Current Crypto Prices

BTC/USDT: HTTP 422: [{'type': 'missing', 'loc': ['body', 'exchange_id'], 'msg': 'Field required', 'input': {'exchange': 'binance', 'symbol': 'BTC/USDT'}, 'url': 'https://errors.pydantic.dev/2.12/v/missing'}] [request_id: 62f560a4-30d7-4604-b7e1-c1d6ad33d281]

ETH/USDT: HTTP 422: [{'type': 'missing', 'loc': ['body', 'exchange_id'], 'msg': 'Field required', 'input': {'exchange': 'binance', 'symbol': 'ETH/USDT'}, 'url': 'https://errors.pydantic.dev/2.12/v/missing'}] [request_id: 32aa9441-204c-46c3-9459-a8f93594912b]

SOL/USDT: HTTP 422: [{'type': 'missing', 'loc': ['body', 'exchange_id'], 'msg': 'Field required', 'input': {'exchange': 'binance', 'symbol': 'SOL/USDT'}, 'url': 'https://errors.pydantic.dev/2.12/v/missing'}] [request_id: 0bb3ecf7-4d06-4cb5-b816-45361ee3bad4]


## 4. Order Book

In [ ]:
# Get L2 order book
try:
    response = qj.ccxt.get_l2_order_book(
        exchange="binance",
        symbol="BTC/USDT",
        limit=10
    )
    orderbook = response.get('value', response) if isinstance(response, dict) else response
    
    if orderbook and isinstance(orderbook, dict):
        bids = orderbook.get('bids', [])
        asks = orderbook.get('asks', [])
        
        print("BTC/USDT Order Book (Top 10)")
        print("=" * 50)
        print(f"{'Bid Price':>15} {'Bid Size':>12} | {'Ask Price':>15} {'Ask Size':>12}")
        print("-" * 50)
        
        for i in range(min(10, len(bids), len(asks))):
            bid_price, bid_size = bids[i] if i < len(bids) else (0, 0)
            ask_price, ask_size = asks[i] if i < len(asks) else (0, 0)
            print(f"${bid_price:>14,.2f} {bid_size:>11,.4f} | ${ask_price:>14,.2f} {ask_size:>11,.4f}")
        
        if bids and asks:
            spread = asks[0][0] - bids[0][0]
            spread_pct = spread / bids[0][0] * 100
            print(f"\nSpread: ${spread:.2f} ({spread_pct:.4f}%)")
except Exception as e:
    print(f"Note: Order book - {e}")


## 5. Recent Trades

In [ ]:
# Get recent trades
try:
    response = qj.ccxt.get_recent_trades(
        exchange="binance",
        symbol="BTC/USDT",
        limit=20
    )
    trades = response.get('value', response) if isinstance(response, dict) else response
    
    if trades:
        df_trades = pd.DataFrame(trades)
        if 'timestamp' in df_trades.columns:
            df_trades['time'] = pd.to_datetime(df_trades['timestamp'], unit='ms')
        
        print("BTC/USDT Recent Trades")
        print("=" * 60)
        
        cols = ['time', 'price', 'amount', 'side']
        available_cols = [c for c in cols if c in df_trades.columns]
        if available_cols:
            print(df_trades[available_cols].tail(10).to_string(index=False))
        else:
            print(df_trades.tail(10))
except Exception as e:
    print(f"Note: Recent trades - {e}")


## 6. Multi-Exchange Comparison

In [ ]:
# Compare BTC price across exchanges
exchanges = ['binance', 'kraken', 'coinbase']
prices = {}

print("BTC/USDT Price Across Exchanges")
print("=" * 50)

for exchange in exchanges:
    try:
        symbol = 'BTC/USDT' if exchange != 'coinbase' else 'BTC/USD'
        response = qj.ccxt.get_ticker(
            exchange=exchange,
            symbol=symbol
        )
        ticker = response.get('value', response) if isinstance(response, dict) else response
        
        if ticker and isinstance(ticker, dict):
            price = ticker.get('last', ticker.get('close', 0))
            prices[exchange] = price
            print(f"{exchange:>12}: ${price:,.2f}")
    except Exception as e:
        print(f"{exchange:>12}: {e}")

if len(prices) > 1:
    max_price = max(prices.values())
    min_price = min(prices.values())
    arb = max_price - min_price
    arb_pct = arb / min_price * 100
    print(f"\nArbitrage opportunity: ${arb:.2f} ({arb_pct:.3f}%)")


## 7. Technical Indicators

In [ ]:
# Add technical indicators to BTC
if 'df_btc' in dir() and len(df_btc) > 0:
    # Moving averages
    df_btc['SMA_20'] = df_btc['close'].rolling(20).mean()
    df_btc['SMA_50'] = df_btc['close'].rolling(50).mean()
    df_btc['SMA_200'] = df_btc['close'].rolling(200).mean()
    
    # RSI
    delta = df_btc['close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
    rs = gain / loss
    df_btc['RSI'] = 100 - (100 / (1 + rs))
    
    # Plot with indicators
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        vertical_spacing=0.1,
                        row_heights=[0.7, 0.3])
    
    # Price with MAs
    fig.add_trace(go.Candlestick(
        x=df_btc['date'],
        open=df_btc['open'],
        high=df_btc['high'],
        low=df_btc['low'],
        close=df_btc['close'],
        name='BTC'
    ), row=1, col=1)
    
    fig.add_trace(go.Scatter(x=df_btc['date'], y=df_btc['SMA_20'],
                             name='SMA 20', line=dict(color='cyan', width=1)), row=1, col=1)
    fig.add_trace(go.Scatter(x=df_btc['date'], y=df_btc['SMA_50'],
                             name='SMA 50', line=dict(color='yellow', width=1)), row=1, col=1)
    
    # RSI
    fig.add_trace(go.Scatter(x=df_btc['date'], y=df_btc['RSI'],
                             name='RSI', line=dict(color='purple')), row=2, col=1)
    fig.add_hline(y=70, line_dash='dash', line_color='red', row=2, col=1)
    fig.add_hline(y=30, line_dash='dash', line_color='green', row=2, col=1)
    
    fig.update_layout(
        title='BTC/USDT with Technical Indicators',
        template='plotly_dark',
        height=700,
        xaxis_rangeslider_visible=False
    )
    fig.show()
    
    # Current signals
    current_price = df_btc['close'].iloc[-1]
    current_rsi = df_btc['RSI'].iloc[-1]
    sma20 = df_btc['SMA_20'].iloc[-1]
    sma50 = df_btc['SMA_50'].iloc[-1]
    
    print(f"\nCurrent Signals:")
    print(f"  Price: ${current_price:,.0f}")
    print(f"  RSI:   {current_rsi:.1f} {'(Overbought)' if current_rsi > 70 else '(Oversold)' if current_rsi < 30 else '(Neutral)'}")
    print(f"  vs SMA20: {'Above' if current_price > sma20 else 'Below'} (${sma20:,.0f})")
    print(f"  vs SMA50: {'Above' if current_price > sma50 else 'Below'} (${sma50:,.0f})")


## Summary

CCXT connector capabilities:
- **OHLCV Data**: Historical candlestick data
- **Ticker**: Real-time prices and 24h stats
- **Order Book**: L2 depth data
- **Trades**: Recent trade history
- **Multi-Exchange**: Compare across 100+ exchanges

### Supported Exchanges:
Binance, Kraken, Coinbase, Bybit, OKX, KuCoin, Huobi, Gate.io, and many more.

### Use Cases:
- Crypto market analysis
- Arbitrage detection
- Technical analysis
- Portfolio tracking

# QuantJourney SDK - CCXT Cryptocurrency Data

This notebook demonstrates CCXT connector:
- Real-time crypto prices from 100+ exchanges
- OHLCV candlestick data
- Multi-exchange comparison
- Order book and trades

**API:** https://api.quantjourney.cloud

In [ ]:
# Setup
import sys
sys.path.insert(0, '..')

from quantjourney.sdk import QuantJourneyAPI
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# API Key authentication
import os
API_KEY = os.environ.get("QJ_API_KEY", "qj_...")
qj = QuantJourneyAPI(api_key=API_KEY)
print("✓ Connected to QuantJourney API")


## 1. Bitcoin OHLCV Data

In [ ]:
# Fetch Bitcoin OHLCV from Binance
try:
    response = qj.ccxt.get_ohlcv_data(
        exchange="binance",
        symbol="BTC/USDT",
        timeframe="1d",
        limit=365
    )
    btc = response.get('value', response) if isinstance(response, dict) else response
    
    if btc:
        df_btc = pd.DataFrame(btc, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
        df_btc['date'] = pd.to_datetime(df_btc['timestamp'], unit='ms')
        df_btc = df_btc.sort_values('date')
        
        print(f"BTC records: {len(df_btc)}")
        print(f"Date range: {df_btc['date'].min().date()} to {df_btc['date'].max().date()}")
        print(f"\nLatest data:")
        print(df_btc[['date', 'open', 'high', 'low', 'close', 'volume']].tail())
except Exception as e:
    print(f"Note: CCXT connector - {e}")


In [ ]:
# Plot Bitcoin candlestick chart
if 'df_btc' in dir() and len(df_btc) > 0:
    fig = go.Figure(data=[
        go.Candlestick(
            x=df_btc['date'],
            open=df_btc['open'],
            high=df_btc['high'],
            low=df_btc['low'],
            close=df_btc['close'],
            name='BTC'
        )
    ])
    
    fig.update_layout(
        title='Bitcoin (BTC/USDT) - Daily Candlestick',
        yaxis_title='Price (USDT)',
        xaxis_rangeslider_visible=False,
        template='plotly_dark',
        height=500
    )
    fig.show()
    
    print(f"\nBTC Statistics:")
    print(f"  Current:  ${df_btc['close'].iloc[-1]:,.0f}")
    print(f"  52w High: ${df_btc['high'].max():,.0f}")
    print(f"  52w Low:  ${df_btc['low'].min():,.0f}")
    print(f"  Avg Vol:  {df_btc['volume'].mean():,.0f} BTC")


## 2. Multi-Crypto Comparison

In [ ]:
# Compare multiple cryptocurrencies
cryptos = ['BTC/USDT', 'ETH/USDT', 'SOL/USDT', 'BNB/USDT', 'XRP/USDT']
crypto_data = {}

print("Fetching crypto data...")
for symbol in cryptos:
    try:
        response = qj.ccxt.get_ohlcv_data(
            exchange="binance",
            symbol=symbol,
            timeframe="1d",
            limit=180
        )
        data = response.get('value', response) if isinstance(response, dict) else response
        if data:
            df = pd.DataFrame(data, columns=['ts', 'o', 'h', 'l', 'c', 'v'])
            df['date'] = pd.to_datetime(df['ts'], unit='ms')
            df = df.set_index('date')
            crypto_data[symbol.split('/')[0]] = df['c']
            print(f"  ✓ {symbol}: ${df['c'].iloc[-1]:,.2f}")
    except Exception as e:
        print(f"  ✗ {symbol}: {e}")


In [ ]:
# Normalized performance comparison
if crypto_data:
    crypto_df = pd.DataFrame(crypto_data).dropna()
    
    # Normalize to 100
    normalized = crypto_df / crypto_df.iloc[0] * 100
    
    fig = px.line(
        normalized,
        title='Crypto Performance Comparison (Normalized to 100)',
        labels={'value': 'Performance', 'variable': 'Crypto'},
        template='plotly_dark'
    )
    fig.update_layout(height=500)
    fig.show()
    
    # Performance summary
    print("\n180-Day Performance:")
    for coin in normalized.columns:
        perf = normalized[coin].iloc[-1] - 100
        print(f"  {coin:>5}: {perf:+.1f}%")


## 3. Current Prices (Ticker)

In [ ]:
# Get current ticker data
symbols = ['BTC/USDT', 'ETH/USDT', 'SOL/USDT']

print("Current Crypto Prices")
print("=" * 50)

for symbol in symbols:
    try:
        response = qj.ccxt.get_ticker(
            exchange="binance",
            symbol=symbol
        )
        ticker = response.get('value', response) if isinstance(response, dict) else response
        
        if ticker and isinstance(ticker, dict):
            last = ticker.get('last', ticker.get('close', 0))
            bid = ticker.get('bid', 0)
            ask = ticker.get('ask', 0)
            volume = ticker.get('baseVolume', ticker.get('volume', 0))
            change = ticker.get('percentage', ticker.get('change', 0))
            
            print(f"\n{symbol}:")
            print(f"  Last:   ${last:,.2f}")
            print(f"  Bid:    ${bid:,.2f}")
            print(f"  Ask:    ${ask:,.2f}")
            print(f"  24h Vol: {volume:,.0f}")
            if change:
                print(f"  24h Chg: {change:+.2f}%")
    except Exception as e:
        print(f"\n{symbol}: {e}")


## 4. Order Book

In [ ]:
# Get L2 order book
try:
    response = qj.ccxt.get_l2_order_book(
        exchange="binance",
        symbol="BTC/USDT",
        limit=10
    )
    orderbook = response.get('value', response) if isinstance(response, dict) else response
    
    if orderbook and isinstance(orderbook, dict):
        bids = orderbook.get('bids', [])
        asks = orderbook.get('asks', [])
        
        print("BTC/USDT Order Book (Top 10)")
        print("=" * 50)
        print(f"{'Bid Price':>15} {'Bid Size':>12} | {'Ask Price':>15} {'Ask Size':>12}")
        print("-" * 50)
        
        for i in range(min(10, len(bids), len(asks))):
            bid_price, bid_size = bids[i] if i < len(bids) else (0, 0)
            ask_price, ask_size = asks[i] if i < len(asks) else (0, 0)
            print(f"${bid_price:>14,.2f} {bid_size:>11,.4f} | ${ask_price:>14,.2f} {ask_size:>11,.4f}")
        
        if bids and asks:
            spread = asks[0][0] - bids[0][0]
            spread_pct = spread / bids[0][0] * 100
            print(f"\nSpread: ${spread:.2f} ({spread_pct:.4f}%)")
except Exception as e:
    print(f"Note: Order book - {e}")


## 5. Recent Trades

In [ ]:
# Get recent trades
try:
    response = qj.ccxt.get_recent_trades(
        exchange="binance",
        symbol="BTC/USDT",
        limit=20
    )
    trades = response.get('value', response) if isinstance(response, dict) else response
    
    if trades:
        df_trades = pd.DataFrame(trades)
        if 'timestamp' in df_trades.columns:
            df_trades['time'] = pd.to_datetime(df_trades['timestamp'], unit='ms')
        
        print("BTC/USDT Recent Trades")
        print("=" * 60)
        
        cols = ['time', 'price', 'amount', 'side']
        available_cols = [c for c in cols if c in df_trades.columns]
        if available_cols:
            print(df_trades[available_cols].tail(10).to_string(index=False))
        else:
            print(df_trades.tail(10))
except Exception as e:
    print(f"Note: Recent trades - {e}")


## 6. Multi-Exchange Comparison

In [ ]:
# Compare BTC price across exchanges
exchanges = ['binance', 'kraken', 'coinbase']
prices = {}

print("BTC/USDT Price Across Exchanges")
print("=" * 50)

for exchange in exchanges:
    try:
        symbol = 'BTC/USDT' if exchange != 'coinbase' else 'BTC/USD'
        response = qj.ccxt.get_ticker(
            exchange=exchange,
            symbol=symbol
        )
        ticker = response.get('value', response) if isinstance(response, dict) else response
        
        if ticker and isinstance(ticker, dict):
            price = ticker.get('last', ticker.get('close', 0))
            prices[exchange] = price
            print(f"{exchange:>12}: ${price:,.2f}")
    except Exception as e:
        print(f"{exchange:>12}: {e}")

if len(prices) > 1:
    max_price = max(prices.values())
    min_price = min(prices.values())
    arb = max_price - min_price
    arb_pct = arb / min_price * 100
    print(f"\nArbitrage opportunity: ${arb:.2f} ({arb_pct:.3f}%)")


## 7. Technical Indicators

In [ ]:
# Add technical indicators to BTC
if 'df_btc' in dir() and len(df_btc) > 0:
    # Moving averages
    df_btc['SMA_20'] = df_btc['close'].rolling(20).mean()
    df_btc['SMA_50'] = df_btc['close'].rolling(50).mean()
    df_btc['SMA_200'] = df_btc['close'].rolling(200).mean()
    
    # Bollinger Bands
    df_btc['BB_mid'] = df_btc['close'].rolling(20).mean()
    df_btc['BB_std'] = df_btc['close'].rolling(20).std()
    df_btc['BB_upper'] = df_btc['BB_mid'] + 2 * df_btc['BB_std']
    df_btc['BB_lower'] = df_btc['BB_mid'] - 2 * df_btc['BB_std']
    
    # RSI
    delta = df_btc['close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
    rs = gain / loss
    df_btc['RSI'] = 100 - (100 / (1 + rs))
    
    # Plot with indicators
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        vertical_spacing=0.1,
                        row_heights=[0.7, 0.3])
    
    # Price with MAs
    fig.add_trace(go.Candlestick(
        x=df_btc['date'],
        open=df_btc['open'],
        high=df_btc['high'],
        low=df_btc['low'],
        close=df_btc['close'],
        name='BTC'
    ), row=1, col=1)
    
    fig.add_trace(go.Scatter(x=df_btc['date'], y=df_btc['SMA_20'],
                             name='SMA 20', line=dict(color='cyan', width=1)), row=1, col=1)
    fig.add_trace(go.Scatter(x=df_btc['date'], y=df_btc['SMA_50'],
                             name='SMA 50', line=dict(color='yellow', width=1)), row=1, col=1)
    
    # RSI
    fig.add_trace(go.Scatter(x=df_btc['date'], y=df_btc['RSI'],
                             name='RSI', line=dict(color='purple')), row=2, col=1)
    fig.add_hline(y=70, line_dash='dash', line_color='red', row=2, col=1)
    fig.add_hline(y=30, line_dash='dash', line_color='green', row=2, col=1)
    
    fig.update_layout(
        title='BTC/USDT with Technical Indicators',
        template='plotly_dark',
        height=700,
        xaxis_rangeslider_visible=False
    )
    fig.show()
    
    # Current signals
    current_price = df_btc['close'].iloc[-1]
    current_rsi = df_btc['RSI'].iloc[-1]
    sma20 = df_btc['SMA_20'].iloc[-1]
    sma50 = df_btc['SMA_50'].iloc[-1]
    
    print(f"\nCurrent Signals:")
    print(f"  Price: ${current_price:,.0f}")
    print(f"  RSI:   {current_rsi:.1f} {'(Overbought)' if current_rsi > 70 else '(Oversold)' if current_rsi < 30 else '(Neutral)'}")
    print(f"  vs SMA20: {'Above' if current_price > sma20 else 'Below'} (${sma20:,.0f})")
    print(f"  vs SMA50: {'Above' if current_price > sma50 else 'Below'} (${sma50:,.0f})")


## Summary

CCXT connector capabilities:
- **OHLCV Data**: Historical candlestick data
- **Ticker**: Real-time prices and 24h stats
- **Order Book**: L2 depth data
- **Trades**: Recent trade history
- **Multi-Exchange**: Compare across 100+ exchanges

### Supported Exchanges:
Binance, Kraken, Coinbase, Bybit, OKX, KuCoin, Huobi, Gate.io, and many more.

### Use Cases:
- Crypto market analysis
- Arbitrage detection
- Technical analysis
- Portfolio tracking